# EDA: Agent Conversation Log

## Setup
Reset database and seed base data.

In [1]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")


Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with ID: 1 and GUID: be649382-cda5-4a20-b3ba-775e1612b06e
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: b754fa1d-8cf6-4475-af00-6f9f959ec49d
Seeded SystemPrompt 'format' with ID: 1 and GUID: 03b2306d-5475-46dd-aec8-7072aea4ae69
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: d9642737-0cef-4257-953b-cc00a23bd561
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [2]:
# 🧪 Run Program Using SessionConfig

from app.db.models import SessionConfig
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Prepare test file
file = Path("tests/session_linked_execution.py")
file.write_text("def ping( user ):\n return f\"pong {user}\"")

# Construct input using session metadata
input_data = {
    "file_name":  str(file),
    "session_id": session_row.id,
    "system":     session_row.name
}

# Run associated program
program = ProgramProviderFactory.create(id=session_row.program_provider_id)
result = program.run(input_data, session_id=session_row.id)

print("✅ Final Program Result (via SessionConfig):")
print(json.dumps(result.model_dump(), indent=2))

✅ Final Program Result (via SessionConfig):
{
  "state": "end",
  "previous_state": "preprocessing",
  "state_type": "end",
  "reason": "success",
  "decision": "final",
  "steps": 2,
  "max_steps": 20,
  "summary": "preprocessing complete",
  "output": {
    "state": "end",
    "file_name": "tests\\session_linked_execution.py",
    "working_file": null,
    "session_id": 1,
    "system": "codecritic_test_session",
    "reason": "preprocessing complete",
    "steps": 2,
    "retry_count": 0,
    "_last_state": "preprocessing",
    "output": {}
  },
  "provider_name": "codecritic_program"
}


### Load raw log entries

In [3]:
import sqlite3
import pandas as pd
from app.db.connection import DB_PATH

# Connect and load
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM agent_conversation_log WHERE session_id='1'", conn)

# Format and print log entries
for i, row in df.iterrows():
    print(f"🔹 Log ID: {row['id']}")
    print(f"📅 Timestamp: {row['timestamp']}")
    print(f"🧠 Agent ID: {row['agent_provider_config_id']}")
    print(f"🔧 Agent Type: {row['agent_type']}")
    print(f"🗣️ Log Entry:\n{row['content']}")
    print("─" * 60)


🔹 Log ID: 1
📅 Timestamp: 2025-06-03T23:12:03.553941+00:00
🧠 Agent ID: 4
🔧 Agent Type: stability
🗣️ Log Entry:
Failed stability checks: {'utf8_valid': 1.0, 'syntax_ok': 1.0, 'can_compile': 1.0, 'py_compile_ok': 1.0, 'can_import': 1.0, 'formatter_idempotent': 0.0, 'symbol_graph_valid': 0.0, 'mypy_ok': 0.0}
────────────────────────────────────────────────────────────


### Parse enum fields

In [ ]:
from app.enums import logging_enums, fsm_enums, agent_enums
print('Columns:', df.columns.tolist())

### Validate field values

In [ ]:
print(df.isnull().sum())

### Basic counts

In [ ]:
print(df.shape)
print(df['timestamp'].min(), df['timestamp'].max())